# 🥙 LSTM on Recipe Data

In this notebook, we'll walk through the steps required to train your own LSTM on the recipes dataset

In [ ]:
%load_ext autoreload
%autoreload 2

import os
import numpy as np
import json
import re
import string
from collections import Counter

import torch
from torch import nn
from torch.utils.data import DataLoader
from transformers import Trainer, TrainingArguments, TrainerCallback

## 0. Parameters <a name="parameters"></a>

In [ ]:
VOCAB_SIZE = 10000
MAX_LEN = 200
EMBEDDING_DIM = 100
N_UNITS = 128
VALIDATION_SPLIT = 0.2
SEED = 42
LOAD_MODEL = False
BATCH_SIZE = 32
EPOCHS = 25

## 1. Load the data <a name="load"></a>

In [ ]:
# Load the full dataset
with open("../../../data/epirecipes/full_format_recipes.json") as json_data:
    recipe_data = json.load(json_data)

In [ ]:
# Filter the dataset
filtered_data = [
    "Recipe for " + x["title"] + " | " + " ".join(x["directions"])
    for x in recipe_data
    if "title" in x
    and x["title"] is not None
    and "directions" in x
    and x["directions"] is not None
]

In [ ]:
# Count the recipes
n_recipes = len(filtered_data)
print(f"{n_recipes} recipes loaded")

In [ ]:
example = filtered_data[9]
print(example)

## 2. Tokenise the data

In [ ]:
# Pad the punctuation, to treat them as separate 'words'
def pad_punctuation(s):
    s = re.sub(f"([{string.punctuation}])", r" \1 ", s)
    s = re.sub(" +", " ", s)
    return s


text_data = [pad_punctuation(x) for x in filtered_data]

In [ ]:
# Display an example of a recipe
example_data = text_data[9]
example_data

In [ ]:
# A DataLoader can batch and shuffle any Python list - here, a list of strings
text_ds = DataLoader(text_data, batch_size=BATCH_SIZE, shuffle=True)

In [ ]:
# Create a vectorisation "layer". PyTorch has no built-in equivalent of Keras'
# TextVectorization, so we write a small one: lowercase, split on whitespace,
# map each word to an integer id and pad / truncate to a fixed length.
class TextVectorizer:
    def __init__(self, max_tokens, output_sequence_length):
        self.max_tokens = max_tokens
        self.output_sequence_length = output_sequence_length

    def split(self, text):
        return text.lower().split()

    def adapt(self, dataset):
        # Count every word, then keep the most frequent ones.
        # Index 0 is reserved for padding ("") and 1 for unknown words ("[UNK]"),
        # exactly like the Keras layer
        counts = Counter()
        for batch in dataset:
            for text in batch:
                counts.update(self.split(text))
        most_common = [word for word, _ in counts.most_common(self.max_tokens - 2)]
        self.vocab = ["", "[UNK]"] + most_common
        self.word_to_index = {word: i for i, word in enumerate(self.vocab)}

    def get_vocabulary(self):
        return self.vocab

    def encode(self, text):
        ids = [self.word_to_index.get(w, 1) for w in self.split(text)]
        ids = ids[: self.output_sequence_length]  # truncate...
        ids += [0] * (self.output_sequence_length - len(ids))  # ...or pad with 0
        return ids

    def __call__(self, texts):
        # One string -> 1D tensor, a list of strings -> 2D tensor (batch, length)
        if isinstance(texts, str):
            return torch.tensor(self.encode(texts))
        return torch.tensor([self.encode(t) for t in texts])


vectorize_layer = TextVectorizer(
    max_tokens=VOCAB_SIZE,
    output_sequence_length=MAX_LEN + 1,
)

In [ ]:
# Adapt the layer to the training set
vectorize_layer.adapt(text_ds)
vocab = vectorize_layer.get_vocabulary()

In [ ]:
# Display some token:word mappings
for i, word in enumerate(vocab[:10]):
    print(f"{i}: {word}")

In [ ]:
# Display the same example converted to ints
example_tokenised = vectorize_layer(example_data)
print(example_tokenised.numpy())

## 3. Create the Training Set

In [ ]:
# Create the training set of recipes and the same text shifted by one word.
# The Trainer can use the list of strings directly as its dataset; prepare_inputs is
# its data_collator: it turns a list of strings into one batch, as a dict whose
# keys match the arguments of the model's forward()
def prepare_inputs(texts):
    tokenized_sentences = vectorize_layer(texts)
    x = tokenized_sentences[:, :-1]
    y = tokenized_sentences[:, 1:]
    return {"input_ids": x, "labels": y}


train_ds = text_data

## 4. Build the LSTM <a name="build"></a>

In [ ]:
class LSTMModel(nn.Module):
    def __init__(self, vocab_size, embedding_dim, n_units):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        # batch_first=True: tensors are (batch, sequence, features) like in Keras
        self.lstm = nn.LSTM(embedding_dim, n_units, batch_first=True)
        # No softmax: we output logits and nn.CrossEntropyLoss applies softmax itself
        self.dense = nn.Linear(n_units, vocab_size)
        # Like the Keras version, padding tokens (id 0) are included in the loss, so the
        # model also learns to predict 0 after the end of a recipe (used as a stop token)
        self.loss_fn = nn.CrossEntropyLoss()

    # The Trainer expects the model to compute its own loss when labels are given
    def forward(self, input_ids, labels=None):
        x = self.embedding(input_ids)
        x, _ = self.lstm(x)  # nn.LSTM returns (all outputs, (h_n, c_n))
        logits = self.dense(x)  # (batch, seq_len, VOCAB_SIZE)
        loss = None
        if labels is not None:
            # CrossEntropyLoss wants the class dimension second: (batch, VOCAB_SIZE, seq_len)
            loss = self.loss_fn(logits.transpose(1, 2), labels)
        return {"loss": loss, "logits": logits}


lstm = LSTMModel(VOCAB_SIZE, EMBEDDING_DIM, N_UNITS)
print(lstm)
print(f"Total params: {sum(p.numel() for p in lstm.parameters()):,}")

In [ ]:
if LOAD_MODEL:
    lstm.load_state_dict(torch.load("./models/lstm.pt"))

## 5. Train the LSTM <a name="train"></a>

In [ ]:
# TrainingArguments holds the optimizer and training settings
# (the Trainer uses AdamW; with weight_decay=0 it is plain Adam)
training_args = TrainingArguments(
    output_dir="./checkpoint",
    learning_rate=0.001,  # Keras "adam" default
    lr_scheduler_type="constant",  # Keras keeps the learning rate fixed
    max_grad_norm=0.0,  # the Trainer clips gradients by default; Keras doesn't
    per_device_train_batch_size=BATCH_SIZE,
    num_train_epochs=EPOCHS,
    logging_strategy="epoch",
    save_strategy="epoch",
    save_only_model=True,  # weights only, like save_weights_only=True
    save_total_limit=1,  # keep just the latest checkpoint
    report_to="tensorboard",
    remove_unused_columns=False,  # our batches are already what the model needs
)

In [ ]:
# Create a TextGenerator callback (the Trainer calls on_epoch_end after each epoch)
class TextGenerator(TrainerCallback):
    def __init__(self, model, index_to_word, top_k=10):
        self.model = model
        self.index_to_word = index_to_word
        self.word_to_index = {
            word: index for index, word in enumerate(index_to_word)
        }  # <1>

    def sample_from(self, probs, temperature):  # <2>
        probs = probs ** (1 / temperature)
        probs = probs / np.sum(probs)
        return np.random.choice(len(probs), p=probs), probs

    @torch.no_grad()  # no gradients needed when generating
    def generate(self, start_prompt, max_tokens, temperature):
        self.model.eval()
        model_device = next(self.model.parameters()).device
        start_tokens = [
            self.word_to_index.get(x, 1) for x in start_prompt.split()
        ]  # <3>
        sample_token = None
        info = []
        while len(start_tokens) < max_tokens and sample_token != 0:  # <4>
            x = torch.tensor([start_tokens], device=model_device)
            logits = self.model(x)["logits"]  # <5>
            # Softmax over the last position (float64 so the probs sum to 1 for numpy)
            probs = logits[0, -1].softmax(dim=-1).double().cpu().numpy()
            sample_token, probs = self.sample_from(probs, temperature)  # <6>
            info.append({"prompt": start_prompt, "word_probs": probs})
            start_tokens.append(sample_token)  # <7>
            start_prompt = start_prompt + " " + self.index_to_word[sample_token]
        print(f"\ngenerated text:\n{start_prompt}\n")
        return info

    def on_epoch_end(self, args, state, control, **kwargs):
        self.generate("recipe for", max_tokens=100, temperature=1.0)
        self.model.train()  # generate() switched the model to eval mode

In [ ]:
# Checkpoints are saved by the Trainer (save_strategy="epoch" above)
# TensorBoard logs go to ./logs (set with an environment variable in transformers 5)
os.environ["TENSORBOARD_LOGGING_DIR"] = "./logs"

# Tokenize starting prompt
text_generator = TextGenerator(lstm, vocab)

In [ ]:
trainer = Trainer(
    model=lstm,
    args=training_args,
    train_dataset=train_ds,
    data_collator=prepare_inputs,
    callbacks=[text_generator],
)
trainer.train()

In [ ]:
# Save the final model
torch.save(lstm.state_dict(), "./models/lstm.pt")

## 6. Generate text using the LSTM

In [ ]:
def print_probs(info, vocab, top_k=5):
    for i in info:
        print(f"\nPROMPT: {i['prompt']}")
        word_probs = i["word_probs"]
        p_sorted = np.sort(word_probs)[::-1][:top_k]
        i_sorted = np.argsort(word_probs)[::-1][:top_k]
        for p, i in zip(p_sorted, i_sorted):
            print(f"{vocab[i]}:   \t{np.round(100*p,2)}%")
        print("--------\n")

In [ ]:
info = text_generator.generate(
    "recipe for roasted vegetables | chop 1 /", max_tokens=10, temperature=1.0
)

In [ ]:
print_probs(info, vocab)

In [ ]:
info = text_generator.generate(
    "recipe for roasted vegetables | chop 1 /", max_tokens=10, temperature=0.2
)

In [ ]:
print_probs(info, vocab)

In [ ]:
info = text_generator.generate(
    "recipe for chocolate ice cream |", max_tokens=7, temperature=1.0
)
print_probs(info, vocab)

In [ ]:
info = text_generator.generate(
    "recipe for chocolate ice cream |", max_tokens=7, temperature=0.2
)
print_probs(info, vocab)